# Fixed-γ analysis over the W bank (set 1)

Based on `fixed_gamma/Gamma_frustration_anlaysis_V2.ipynb`, rewritten for the per-W files in `gamma_data_V3`. One γ at a time.

**Three kinds of plots**
- **Averaged over all Ws** (pass 1, streamed, threaded, cached): flips, tissue frustration and energy vs sweep; final energy; number of unique profiles; *average* intracellular and intercellular frustration violins (each sim's tissue mean, averaged over the W's sims, gives one value per W).
- **A fixed set of `n_overlay` Ws** (pass 2): *cell* frustration distribution violins. All cells of a W's 10 sims are pooled into one distribution per W, and the `n_overlay` Ws are overlaid. They are picked at random with `overlay_seed`, so they are the same every run.
- **One simulation** (`single_W`, `single_sim`): energy trajectories of that W's sims, final tissue pattern, intracellular and intercellular frustration heatmaps. Tissue patterns differ between runs, so these are never averaged.

**Definitions** (unchanged, checked cell-by-cell against `intracellular_frustration_density` and `intercellular_frustration_density` on test files; all identical):
- *intracellular*: fraction of the cell's own GRN interactions `K[A,B] σ_A σ_B` that are negative;
- *intercellular*: fraction of the cell-cell couplings `W[A,B] σ_iA σ_jB` with its 8 neighbours that are negative. The coupling is scaled by γ, as in the model definition. For γ = 0 all intercellular couplings are zero, so intercellular frustration is zero by definition.

The W-bank runs start from random states only (`σrand`), so the two-state sections of the original are not included. UMAP is left out.

**Threads:** `Threads.nthreads()` is printed below. With 1 thread it works, just more slowly (see the γ-range V2 notebook for setting up a threaded kernel).

In [ ]:
using Plots
using StatsPlots
using StatsBase
using Statistics
using Random
using JLD2
using CodecZlib   # older JLD2 versions need this to read compressed files

# Lattice (neighbour lists) for the frustration calculations
include("../../../Model/opti/functions/ising_core_opti.jl")
# create_configuration_color_key, configuration_id (tissue pattern colours)
include("../../OG/functions/functions_plotting_OG.jl")

println("Julia threads: ", Threads.nthreads())

In [ ]:
### ----------- data location ----------- ###
base_path = "/data/scratch/projects/punim0638/woodward/nonrcep_casestudy/gamma_data_V3"
cache_dir = "/home/twwoodw/my_folder/compute/data/gamma_study"

function make_filename(base_path, W_name, β_name, σ_name, n_sims, max_sweeps)
    return joinpath(base_path,
        "$(W_name)_$(β_name)_$(σ_name)_$(n_sims)_max$(max_sweeps)sweeps_flipequ.jld2")
end

### ----------- constants ----------- ###
n_cells    = 400
n_genes    = 9
n_sims     = 10
max_sweeps = 10_000

β_name = "betaInf"
σ_name = "σrand"

grns = ["tree", "negtri", "ICFF", "CFF", "negtritwin", "negtritri"]

### ----------- fixed γ ----------- ###
γ      = 0.05
γ_name = "g005"          # must match the group name in the files

### ----------- which Ws ----------- ###
W_range = 1:1000
W_signs = [1]            # [1] = W only, [-1] = -W only, [1, -1] = both

n_overlay    = 20        # Ws whose cell distributions are overlaid in the violin plots
overlay_seed = 1         # same seed → same Ws every run

single_W   = "W1"        # the one simulation used for tissue / heatmap / trajectory plots
single_sim = 1

cache_file = joinpath(cache_dir, "summary_gammafixed_$(γ_name)_set1_V4.jld2")

In [ ]:
####################################################################
# Frustration of every cell (same definitions as the original notebook)
####################################################################
"""
intracellular_cell_frustration!(out, σ, K_pairs)

Intracellular frustration density of every cell: the fraction of the cell's own GRN
interactions K[A,B] σ_A σ_B (A ≠ B, K[A,B] ≠ 0) that are negative.
Same value as `intracellular_frustration_density(K, σ[i, :])` / `eval_KGRN`.
`K_pairs = grn_pairs(K)`.
"""
grn_pairs(K) = [(A, B, Int(sign(K[A, B]))) for A in axes(K, 1), B in axes(K, 2) if A != B && !iszero(K[A, B])]

function intracellular_cell_frustration!(out::Vector{Float64}, σ::AbstractMatrix, K_pairs)
    n_nz = length(K_pairs)
    @inbounds for c in axes(σ, 1)
        n_unsat = 0
        for (A, B, s) in K_pairs
            n_unsat += s * Int(σ[c, A]) * Int(σ[c, B]) < 0
        end
        out[c] = n_unsat / n_nz          # a GRN with no edges gives NaN, as in the original
    end
    return out
end

"""
intercellular_cell_frustration!(out, σ, sW, lat, nsum)

Intercellular frustration density of every cell: the fraction of the cell-cell couplings
γ J[i,j] W[A,B] σ_iA σ_jB between cell i and its 8 neighbours that are negative.
Same value as `intercellular_frustration_density(i, state_dept_frustration_idx_faster(J_sys, σ))`
with J_sys = I ⊗ K + J ⊗ (γW). For γ > 0 the
sign, and so the frustration, does not depend on γ).

Only signs matter, so with p = sign(W_AB) σ_A σ_B ∈ {±1} and Z = number of nonzero W_AB:
n_unsat(i) = (n_nonzero(i) - Σp) / 2, where Σp = σ_iᵀ · sign(W) · (Σ_{neighbours j} σ_j).
Whole-number arithmetic, so the result is exact. `sW = sign.(γ .* W)`, `nsum` is an
n_genes × n_cells Int work buffer.
"""
function intercellular_cell_frustration!(out::Vector{Float64}, σ::AbstractMatrix, sW::AbstractMatrix, lat, nsum::Matrix{Int})
    n_cells, G = size(σ)
    Z = count(!iszero, sW)
    fill!(nsum, 0)
    @inbounds for c in 1:n_cells
        nb = lat.frus_nbr[c]; nJ = lat.frus_J[c]
        n_nb = 0
        for k in eachindex(nb)
            iszero(nJ[k]) && continue
            n_nb += 1
            c2 = nb[k]
            for B in 1:G
                nsum[B, c] += Int(σ[c2, B])
            end
        end
        Σp = 0
        for A in 1:G
            s = 0
            for B in 1:G
                s += Int(sW[A, B]) * nsum[B, c]
            end
            Σp += Int(σ[c, A]) * s
        end
        n_nz = n_nb * Z
        out[c] = n_nz == 0 ? 0.0 : ((n_nz - Σp) ÷ 2) / n_nz
    end
    return out
end

"""Number of unique gene-expression profiles (cell types) in a tissue."""
n_unique_profiles(σ) = length(unique(eachrow(σ)))

"""Mean over the sims still running at each sweep (NaN once all have stopped)."""
function active_mean(M::AbstractMatrix)
    out = fill(NaN, size(M, 1))
    for s in axes(M, 1)
        tot = 0.0; n = 0
        for v in view(M, s, :)
            isnan(v) || (tot += v; n += 1)
        end
        n > 0 && (out[s] = tot / n)
    end
    return out
end

####################################################################
# Pass 1: small numbers for every W (streamed, threaded, cached)
####################################################################
"""
summarize_fixed_gamma_file(file, γ_name, grns, lat, acc; n_sims, max_sweeps)

Read only the `γ_name` group of each GRN in one per-W file and keep, per sim:
final energy, sweeps used, number of unique profiles, and the tissue-mean intracellular
and intercellular frustration of the final tissue. The per-sweep curves (avg flips,
avg tissue frustration, mean energy over running sims) are added into `acc`.
"""
function summarize_fixed_gamma_file(file, γ_name, grns, lat, acc; n_sims, max_sweeps)
    nK = length(grns)
    final_E     = fill(NaN, n_sims, nK)
    sweeps_used = zeros(Int, n_sims, nK)
    n_profiles  = zeros(Int, n_sims, nK)
    intracellular_mean  = fill(NaN, n_sims, nK)
    intercellular_mean = fill(NaN, n_sims, nK)
    curves      = fill(NaN, max_sweeps, 3, nK)     # flips, tissue frustration, energy

    jldopen(file, "r") do f
        f["meta/n_sims"] == n_sims         || error("n_sims = $(f["meta/n_sims"]), expected $n_sims")
        f["meta/max_sweeps"] == max_sweeps || error("max_sweeps = $(f["meta/max_sweeps"]), expected $max_sweeps")
        γ_name in f["meta/gamma_names"]    || error("γ $γ_name not in file")
        W    = f["meta/W"]
        Kmap = f["meta/K"]
        γ = Dict(zip(f["meta/gamma_names"], f["meta/gamma_values"]))[γ_name]
        sW = sign.(γ .* W)
        nsum = zeros(Int, size(W, 1), lat.n_cells)
        buf  = zeros(lat.n_cells)

        for (k, Kn) in enumerate(grns)
            haskey(Kmap, Kn) || error("GRN $Kn not in file")
            grp = f["$(γ_name)/$(Kn)"]
            su = grp["sweeps_used"]
            E  = grp["E_selfish"]
            σf = grp["sigma_finals_selfish"]
            Kp = grn_pairs(Kmap[Kn])

            for sim in 1:n_sims
                final_E[sim, k]     = E[su[sim], sim]
                sweeps_used[sim, k] = su[sim]
                n_profiles[sim, k]  = n_unique_profiles(σf[sim])
                intracellular_mean[sim, k]  = mean(intracellular_cell_frustration!(buf, σf[sim], Kp))
                intercellular_mean[sim, k] = mean(intercellular_cell_frustration!(buf, σf[sim], sW, lat, nsum))
            end

            curves[:, 1, k] .= grp["avg_flip_counts"]
            curves[:, 2, k] .= grp["avg_tissue_frustration"]
            curves[:, 3, k] .= active_mean(E)
        end
    end

    for i in eachindex(curves)
        v = curves[i]
        isnan(v) || (acc.sum[i] += v; acc.cnt[i] += 1)
    end

    return (; final_E, sweeps_used, n_profiles, intracellular_mean, intercellular_mean)
end

new_curve_acc(max_sweeps, nK) = (sum = zeros(max_sweeps, 3, nK), cnt = zeros(Int, max_sweeps, 3, nK))

"""
summarize_fixed_gamma(files; γ_name, grns, n_cells, n_sims, max_sweeps, cache_file = nothing)

Pass 1 over many W files at one γ. Memory does not grow with the number of files beyond a
few hundred bytes per W. Uses all Julia threads. With `cache_file`, later calls only read
files that are not in the cache yet. Files missing the γ / a GRN, or with other settings,
are skipped with a warning.

Returns a NamedTuple; per-sim arrays are [sim, W, GRN] in the order of `S.W_files` and `grns`;
curves are [sweep, GRN] (`flips_*`, `frus_*`, `E_*` sums and counts over Ws).
"""
function summarize_fixed_gamma(files; γ_name, grns, n_cells, n_sims, max_sweeps, cache_file = nothing)
    nK = length(grns)
    settings = Dict("gamma_name" => γ_name, "grns" => collect(grns), "n_cells" => n_cells,
                    "n_sims" => n_sims, "max_sweeps" => max_sweeps, "metric_version" => "V4")

    S_old = nothing
    if cache_file !== nothing && isfile(cache_file)
        d = load(cache_file)
        if d["settings"] == settings
            S_old = (; (Symbol(k) => v for (k, v) in d)...)
        else
            @warn "Cache $cache_file was made with different settings; rebuilding it"
        end
    end
    done_files = S_old === nothing ? Set{String}() : Set(S_old.W_files)
    todo = [f for f in files if !(basename(f) in done_files)]
    println("$(length(files)) files requested: $(length(files) - length(todo)) from cache, ",
            "$(length(todo)) to read on $(Threads.nthreads()) thread(s)")

    lat     = Lattice(n_cells; n_neighbors = 8)
    results = Vector{Any}(nothing, length(todo))
    n_read  = Threads.Atomic{Int}(0)
    t0      = time()
    chunks  = collect(Iterators.partition(eachindex(todo), max(1, cld(length(todo), Threads.nthreads()))))

    accs = fetch.([Threads.@spawn begin
        acc = new_curve_acc(max_sweeps, nK)
        for i in chunk
            try
                results[i] = summarize_fixed_gamma_file(todo[i], γ_name, grns, lat, acc;
                                                        n_sims = n_sims, max_sweeps = max_sweeps)
            catch e
                @warn "Skipping $(basename(todo[i])): $(sprint(showerror, e))"
            end
            k = Threads.atomic_add!(n_read, 1) + 1
            k % 100 == 0 && println("  read $k / $(length(todo)) files ($(round(time() - t0, digits = 1)) s)")
        end
        acc
    end for chunk in chunks])

    ok      = findall(!isnothing, results)
    new     = results[ok]
    W_files = vcat(S_old === nothing ? String[] : S_old.W_files, basename.(todo[ok]))
    isempty(W_files) && error("no usable files")

    stack(field) = cat((S_old === nothing ? [] : [getfield(S_old, field)])...,
                       [reshape(r[field], n_sims, 1, nK) for r in new]...; dims = 2)
    acc0 = S_old === nothing ? new_curve_acc(max_sweeps, nK) : (sum = S_old.curve_sum, cnt = S_old.curve_cnt)

    S = (
        settings    = settings,
        W_files     = W_files,
        W_names     = [first(split(f, "_")) for f in W_files],
        final_E     = stack(:final_E),
        sweeps_used = stack(:sweeps_used),
        n_profiles  = stack(:n_profiles),
        intracellular_mean  = stack(:intracellular_mean),
        intercellular_mean = stack(:intercellular_mean),
        curve_sum   = reduce(+, (a.sum for a in accs); init = acc0.sum),
        curve_cnt   = reduce(+, (a.cnt for a in accs); init = acc0.cnt),
    )

    if cache_file !== nothing && !isempty(todo)
        mkpath(dirname(abspath(cache_file)))
        jldsave(cache_file; pairs(S)...)
        println("Saved summary of $(length(W_files)) files to $cache_file")
    end
    println("Done in $(round(time() - t0, digits = 1)) s")
    return S
end

"""Per-sweep W average of a curve: `which` = :flips, :frus or :energy. Returns (sweeps, values)."""
function W_average_curve(S, which::Symbol, k)
    j = which == :flips ? 1 : which == :frus ? 2 : which == :energy ? 3 : error("which = :flips, :frus or :energy")
    c = view(S.curve_cnt, :, j, k)
    sweeps = findall(>(0), c)
    return sweeps, S.curve_sum[sweeps, j, k] ./ c[sweeps]
end

####################################################################
# Pass 2: full per-cell detail for a few Ws only
####################################################################
"""
cell_detail(file, γ_name, grns, lat; keep_tissues = false)

Per-cell intracellular and intercellular frustration of every final tissue in one W file (n_cells × n_sims
per GRN). With `keep_tissues = true` also returns the final tissues and E_selfish, for the
single-simulation plots.
"""
function cell_detail(file, γ_name, grns, lat; keep_tissues = false)
    jldopen(file, "r") do f
        W    = f["meta/W"]
        Kmap = f["meta/K"]
        γ = Dict(zip(f["meta/gamma_names"], f["meta/gamma_values"]))[γ_name]
        sW = sign.(γ .* W)
        nsum = zeros(Int, size(W, 1), lat.n_cells)
        out  = Dict{String,Any}()
        for Kn in grns
            grp = f["$(γ_name)/$(Kn)"]
            σf  = grp["sigma_finals_selfish"]
            Kp  = grn_pairs(Kmap[Kn])
            intracellular = zeros(lat.n_cells, length(σf))
            intercellular = zeros(lat.n_cells, length(σf))
            for sim in eachindex(σf)
                intracellular[:, sim] = intracellular_cell_frustration!(zeros(lat.n_cells), σf[sim], Kp)
                intercellular[:, sim] =intercellular_cell_frustration!(zeros(lat.n_cells), σf[sim], sW, lat, nsum)
            end
            d = Dict{String,Any}("intracellular" => intracellular, "intercellular" => intercellular)
            if keep_tissues
                d["sigma_finals"] = σf
                d["E_selfish"]    = grp["E_selfish"]
            end
            out[Kn] = d
        end
        return out
    end
end

# Pass 1: summarise all Ws at this γ
The first run reads every file (only this γ's groups). Later runs load `cache_file` and read only new files. Missing files are skipped. Each γ has its own cache file.

In [ ]:
W_name(i, sgn) = sgn == 1 ? "W$(i)" : "-W$(i)"

all_files = [make_filename(base_path, W_name(i, s), β_name, σ_name, n_sims, max_sweeps)
             for i in W_range for s in W_signs]
files = filter(isfile, all_files)
println("$(length(files)) of $(length(all_files)) files exist")

@time S = summarize_fixed_gamma(files;
    γ_name     = γ_name,
    grns       = grns,
    n_cells    = n_cells,
    n_sims     = n_sims,
    max_sweeps = max_sweeps,
    cache_file = cache_file
);

n_W = length(S.W_files)
println("Ws in summary: $n_W | longest sim: $(maximum(S.sweeps_used)) sweeps")

# Pass 2: per-cell detail for `n_overlay` Ws and the single simulation

In [ ]:
lat = Lattice(n_cells; n_neighbors = 8)

overlay_names = sort(shuffle(MersenneTwister(overlay_seed), S.W_names)[1:min(n_overlay, n_W)];
                     by = n -> parse(Int, replace(n, "-" => "", "W" => "")))
overlay_files = [make_filename(base_path, n, β_name, σ_name, n_sims, max_sweeps) for n in overlay_names]
println("overlay Ws: ", join(overlay_names, ", "))

@time overlay = [cell_detail(f, γ_name, grns, lat) for f in overlay_files];   # overlay[w][K_name]["intracellular" / "intercellular"]: n_cells × n_sims

single_file = make_filename(base_path, single_W, β_name, σ_name, n_sims, max_sweeps)
single = cell_detail(single_file, γ_name, grns, lat; keep_tissues = true)  # + "sigma_finals", "E_selfish"

In [ ]:
ki(Kn) = findfirst(==(Kn), grns)

grn_colors = Dict(
    "tree"       => :orange,
    "tri"        => :green,
    "negtri"     => :teal,
    "ICFF"       => :purple,
    "CFF"        => :violet,
    "negtritwin" => :blue,
    "negtritri"  => :cyan
)

L = Int(sqrt(n_cells))
tag = "$(γ_name), $(n_W) Ws"

# Average flips (averaged over Ws)

In [ ]:
plt = plot(xlabel = "Sweeps", ylabel = "Average number of flips",
           title = "Avg number of gene state flips: $(tag)")

for K_name in grns
    sweeps, vals = W_average_curve(S, :flips, ki(K_name))
    plot!(plt, sweeps, vals; label = K_name, color = grn_colors[K_name], alpha = 0.6, linewidth = 2)
end
#xlims!(0, 2000)

display(plt)
#savefig(plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/avg_tissue_flips_$(σ_name)_$(γ_name)_set1_V3.png")

# Average frustration (averaged over Ws)

In [ ]:
plt = plot(xlabel = "Sweeps", ylabel = "Average tissue frustration",
           title = "Average tissue frustration per GRN: $(tag)")

for K_name in grns
    sweeps, vals = W_average_curve(S, :frus, ki(K_name))
    plot!(plt, sweeps, vals; label = K_name, color = grn_colors[K_name], alpha = 0.6, linewidth = 2)
end
#xlims!(0, 500)

display(plt)
#savefig(plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/avg_tissue_frustration_$(σ_name)_$(γ_name)_set1_V3.png")

# Average energy (averaged over Ws)
For each W: mean energy over its sims still running at each sweep (as in the original), then averaged over Ws.

In [ ]:
plt = plot(xlabel = "Sweeps", ylabel = "Average tissue Energy",
           title = "Average tissue Energy per GRN: $(tag)")

for K_name in grns
    sweeps, vals = W_average_curve(S, :energy, ki(K_name))
    plot!(plt, sweeps, vals; label = K_name, color = grn_colors[K_name], alpha = 0.6, linewidth = 2)
end
#xlims!(0, 500)

display(plt)
#savefig(plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/avg_tissue_energy_$(σ_name)_$(γ_name)_set1_V3.png")

# Check for meta-stability and oscillations (single W)
Every simulation of `single_W`, with their mean on top.

In [ ]:
plots_list = []

for K_name in grns
    E = single[K_name]["E_selfish"]
    p = plot(title = "$(K_name) - $(γ_name), $(single_W)", xlabel = "Sweep", ylabel = "Energy", legend = false)

    for sim in 1:size(E, 2)
        valid_idx = findall(!isnan, E[:, sim])
        isempty(valid_idx) && continue
        plot!(p, valid_idx, E[valid_idx, sim]; alpha = 0.3, color = grn_colors[K_name])
    end

    avg = active_mean(E)
    valid_idx = findall(!isnan, avg)
    plot!(p, valid_idx, avg[valid_idx]; color = grn_colors[K_name], linewidth = 2.5)

    push!(plots_list, p)
end

plt = plot(plots_list..., layout = (2, 3), size = (1200, 900))
display(plt)
#savefig(plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/ALL_tissue_energy_tragectories_$(single_W)_$(σ_name)_$(γ_name)_set1_V3.png")

# Final tissue energy (all sims of all Ws)

In [ ]:
plt = plot(xlabel = "GRN", ylabel = "Final Energy", title = "Final Tissue energy: $(tag)", legend = false)

for (k, K_name) in enumerate(grns)
    violin!(plt, fill(k, length(S.final_E[:, :, k])), vec(S.final_E[:, :, k]);
            color = grn_colors[K_name], alpha = 0.75, linewidth = 0)
end
xticks!(plt, 1:length(grns), grns)

display(plt)
#savefig(plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/final_tissue_energy_violin_$(σ_name)_$(γ_name)_set1_V3.png")

# Energy statistical analysis (all sims of all Ws)

In [ ]:
for (k, K_name) in enumerate(grns)
    clean = filter(!isnan, vec(S.final_E[:, :, k]))
    println("$(K_name):  ($(length(clean)) sims)")
    println("  mean:     ", mean(clean))
    println("  median:   ", median(clean))
    println("  std:      ", std(clean))
    println("  skewness: ", skewness(clean))
    println("  kurtosis: ", kurtosis(clean))
    println()
end

# Final tissues (single simulation)

In [ ]:
function final_configuration_plot(
    σ_test;
    sim = single_sim,
    color_key = create_configuration_color_key(9),
    title = "GRN"
)
    σ = σ_test[sim]
    n_cells, n_genes = size(σ)

    L = Int(sqrt(n_cells))
    @assert L * L == n_cells "n_cells must be a perfect square"

    configs = [Tuple(σ[i, :]) for i in 1:n_cells]
    types = [configuration_id(cfg) for cfg in configs]
    n_unique_configs = length(unique(types))

    grid = reshape(types, L, L)

    p = heatmap(
        grid;
        c = color_key,
        clims = (0.5, 2^n_genes + 0.5),
        aspect_ratio = :equal,
        title = "$(title) final tissue - $(n_unique_configs) profiles")

    return p, n_unique_configs
end

color_key = create_configuration_color_key(n_genes);

In [ ]:
tissue_plots = [first(final_configuration_plot(single[K_name]["sigma_finals"];
                                               sim = single_sim, color_key = color_key, title = K_name))
                for K_name in grns]

plt = plot(tissue_plots..., layout = (2, 3), size = (1200, 800),
           plot_title = "$(single_W), sim $(single_sim), $(γ_name)")
display(plt)
#savefig(plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/final_tissues_$(single_W)_sim$(single_sim)_$(σ_name)_$(γ_name)_set1_V3.png")

# Number of unique profiles (averaged over each W's sims, all Ws)

In [ ]:
plt = plot(xlabel = "GRN", ylabel = "Number of unique profiles (mean over sims)",
           title = "Final tissue profiles: $(tag)", legend = false)

for (k, K_name) in enumerate(grns)
    per_W = vec(mean(S.n_profiles[:, :, k]; dims = 1))
    violin!(plt, fill(k, length(per_W)), per_W; color = grn_colors[K_name], alpha = 0.75, linewidth = 0)
end
xticks!(plt, 1:length(grns), grns)

display(plt)
#savefig(plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/final_profile_counts_violin_$(σ_name)_$(γ_name)_set1_V3.png")

# Frustration in tissue (single simulation)

In [ ]:
intercellular_plots = []
intracellular_plots  = []

for K_name in grns
    intercellular_grid = reshape(single[K_name]["intercellular"][:, single_sim], (L, L))
    intracellular_grid  = reshape(single[K_name]["intracellular"][:, single_sim],  (L, L))

    push!(intercellular_plots, heatmap(intercellular_grid; title = "$(K_name) — $(single_W) sim $(single_sim)",
                                aspect_ratio = 1, colorbar = true, clims = (0, 0.7)))
    push!(intracellular_plots,  heatmap(intracellular_grid;  title = "$(K_name) — $(single_W) sim $(single_sim)",
                                aspect_ratio = 1, colorbar = true, clims = (0, 0.7)))
end

In [ ]:
intercellular_plt = plot(intercellular_plots..., layout = (2, 3), size = (1200, 800), plot_title = "intercellular Frustration ($(γ_name))")
display(intercellular_plt)
#savefig(intercellular_plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/intercellular_frustration_mtx_$(single_W)_sim$(single_sim)_$(σ_name)_$(γ_name)_set1_V3.png")

In [ ]:
intracellular_plt = plot(intracellular_plots..., layout = (2, 3), size = (1200, 800), plot_title = "intracellular Frustration ($(γ_name))")
display(intracellular_plt)
#savefig(intracellular_plt, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/intracellular_frustration_mtx_$(single_W)_sim$(single_sim)_$(σ_name)_$(γ_name)_set1_V3.png")

## Violin plots
- **Cell frustration distributions** (plots 1 and 3): one violin per overlay W = all cells of that W's 10 final tissues pooled; the `n_overlay` Ws are overlaid.
- **Average frustration** (plots 2 and 4): each sim's tissue-mean frustration, averaged over the W's sims, gives one value per W. One violin per GRN over **all** Ws.

In [ ]:
# =========================================================
# Plot 1 — intracellular cell frustration distributions (n_overlay Ws)
# =========================================================
p1 = plot(xlabel = "GRN", ylabel = "intracellular cell frustration",
          title = "intracellular cell frustration distributions ($(length(overlay)) Ws)", legend = false)

for (k, K_name) in enumerate(grns), W_data in overlay
    violin!(p1, [k], vec(W_data[K_name]["intracellular"]);
            color = grn_colors[K_name], alpha = 0.5, linewidth = 1, linecolor = :black)
end
xticks!(p1, 1:length(grns), grns)

In [ ]:
# =========================================================
# Plot 2 — Average intracellular frustration (all Ws)
# =========================================================
p2 = plot(xlabel = "GRN", ylabel = "Average intracellular cell frustration",
          title = "Average intracellular cell frustration ($(n_W) Ws)", legend = false)

for (k, K_name) in enumerate(grns)
    per_W = vec(mean(S.intracellular_mean[:, :, k]; dims = 1))
    violin!(p2, fill(k, length(per_W)), per_W; color = grn_colors[K_name], linewidth = 0)
end
xticks!(p2, 1:length(grns), grns)

In [ ]:
# =========================================================
# Plot 3 — intercellular cell frustration distributions (n_overlay Ws)
# =========================================================
p3 = plot(xlabel = "GRN", ylabel = "intercellular cell frustration density",
          title = "intercellular cell frustration distributions ($(length(overlay)) Ws)", legend = false)

for (k, K_name) in enumerate(grns), W_data in overlay
    violin!(p3, [k], vec(W_data[K_name]["intercellular"]);
            color = grn_colors[K_name], alpha = 0.5, linewidth = 1, linecolor = :black)
end
xticks!(p3, 1:length(grns), grns)

In [ ]:
# =========================================================
# Plot 4 — Average intercellular frustration (all Ws)
# =========================================================
p4 = plot(xlabel = "GRN", ylabel = "Average cell intercellular frustration",
          title = "Average cell intercellular frustration ($(n_W) Ws)", legend = false)

for (k, K_name) in enumerate(grns)
    per_W = vec(mean(S.intercellular_mean[:, :, k]; dims = 1))
    violin!(p4, fill(k, length(per_W)), per_W; color = grn_colors[K_name], linewidth = 0)
end
xticks!(p4, 1:length(grns), grns)

In [ ]:
final_plot = plot(p1, p2, p3, p4, layout = (2, 2), size = (1200, 900), dpi = 600)
display(final_plot)
#savefig(final_plot, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/frustration_violin_$(σ_name)_$(γ_name)_set1_V3.png")

In [ ]:
# =========================================================
# Plot 5 — Intracellular vs Intercellular average frustration (all Ws)
# =========================================================
p5 = plot(xlabel = "GRN", ylabel = "Mean frustration",
          title = "Intracellular vs Intercellular frustration ($(n_W) Ws)", legend = :topright)

for (k, K_name) in enumerate(grns)
    intracellular_W  = vec(mean(S.intracellular_mean[:, :, k];  dims = 1))
    intercellular_W = vec(mean(S.intercellular_mean[:, :, k]; dims = 1))

    violin!(p5, fill(k - 0.15, length(intracellular_W)), intracellular_W / 8 * 9;   # scaling kept from the original notebook
            color = grn_colors[K_name], alpha = 0.75, linewidth = 1, linecolor = :black,
            label = k == 1 ? "intracellular" : "")
    violin!(p5, fill(k + 0.15, length(intercellular_W)), intercellular_W;
            color = grn_colors[K_name], linewidth = 0,
            label = k == 1 ? "intercellular" : "")
end
xticks!(p5, 1:length(grns), grns)
display(p5)

In [ ]:
# =========================================================
# Intracellular vs Intercellular cell frustration distributions (n_overlay Ws)
# =========================================================
p_intracellular_intercellular = plot(xlabel = "GRN", ylabel = "Frustration",
                      title = "Intracellular vs Intercellular frustration distributions ($(length(overlay)) Ws)",
                      legend = :topright, dpi = 600)

for (k, K_name) in enumerate(grns), (w, W_data) in enumerate(overlay)
    violin!(p_intracellular_intercellular, [k - 0.15], vec(W_data[K_name]["intracellular"]);
            color = grn_colors[K_name], alpha = 0.5, linewidth = 1, linecolor = :black,
            label = (k == 1 && w == 1) ? "intracellular" : "")
    violin!(p_intracellular_intercellular, [k + 0.15], vec(W_data[K_name]["intercellular"]);
            color = grn_colors[K_name], alpha = 0.5, linewidth = 0,
            label = (k == 1 && w == 1) ? "intercellular" : "")
end
xticks!(p_intracellular_intercellular, 1:length(grns), grns)
ylims!(p_intracellular_intercellular, 0, 1)
display(p_intracellular_intercellular)
#savefig(p_intracellular_intercellular, "~/my_folder/compute/data/gamma_study/gamma_plots_set1/All_frustration_violin_$(σ_name)_$(γ_name)_set1_V3.png")